# TP 20 — Vision Transformer : visualiser l'attention multi-tête

**Objectifs**

- Charger un Vision Transformer pré-entraîné via `timm`.
- Capturer l'attention du dernier bloc (matrice `softmax(QK^T / √d_k)`).
- Visualiser la spécialisation des têtes en overlay sur l'image.
- Construire une carte de régions d'intérêt par seuillage.

**Durée indicative : 50 minutes.**

**Modèle utilisé :** `vit_small_patch16_224.augreg_in21k_ft_in1k` (ViT-S/16, ~22M paramètres, CPU-friendly).


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import timm
import torch
from PIL import Image
from torchvision import transforms

## Exercice 1 — Charger un ViT pré-entraîné et préparer une image

1. Créez le modèle via `timm.create_model("vit_small_patch16_224.augreg_in21k_ft_in1k", pretrained=True, num_classes=0)`.
   Le `num_classes=0` retire la tête de classification : on garde juste le backbone.
2. Mettez le modèle en mode `eval()` (pas d'entraînement).
3. **Désactivez `fused_attn` sur tous les blocs.** Sans ça, PyTorch utilise une attention fusionnée qui ne matérialise jamais la matrice d'attention — on ne pourra pas la visualiser.
4. Chargez une image au choix (vous pouvez télécharger `https://upload.wikimedia.org/wikipedia/commons/thumb/2/2c/Wiki-cat.jpg/640px-Wiki-cat.jpg` par exemple, ou utiliser un fichier local).
5. Appliquez le preprocessing **identique à celui d'entraînement ImageNet** : `Resize(256)` → `CenterCrop(224)` → `ToTensor` → `Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])`.

<details>
<summary>💡 Coup de pouce — charger un ViT et désactiver fused_attn</summary>

**🎯 But :** se mettre dans des conditions où l'attention sera capturable plus loin.

**Charger le modèle**

```python
model = timm.create_model(
    "vit_small_patch16_224.augreg_in21k_ft_in1k",
    pretrained=True,    # téléchargement automatique depuis HuggingFace
    num_classes=0,      # pas de tête de classif, juste le backbone
)
model.eval()
```

Quelques propriétés utiles à inspecter :

```python
print(model.patch_embed.patch_size)      # (16, 16)
print(model.embed_dim)                    # 384
print(model.blocks[0].attn.num_heads)     # 6
print(len(model.blocks))                  # 12 blocs
```

**Désactiver `fused_attn` (étape critique)**

```python
for blk in model.blocks:
    if hasattr(blk.attn, "fused_attn"):
        blk.attn.fused_attn = False
```

⚠️ **Pourquoi ?** Par défaut, `timm` utilise `torch.nn.functional.scaled_dot_product_attention` qui ne renvoie que la sortie attention(Q,K,V), pas la matrice intermédiaire `softmax(QK^T/√d_k)`. Pour visualiser, on doit retomber sur le chemin "non fusionné" qui matérialise explicitement cette matrice.

**Charger et preprocesser une image**

```python
import urllib.request
url = "https://upload.wikimedia.org/wikipedia/commons/thumb/2/2c/Wiki-cat.jpg/640px-Wiki-cat.jpg"
urllib.request.urlretrieve(url, "sample.jpg")
pil_image = Image.open("sample.jpg").convert("RGB")

IMG_SIZE = 224
preprocess = transforms.Compose([
    transforms.Resize(IMG_SIZE + 32),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
x = preprocess(pil_image).unsqueeze(0)    # (1, 3, 224, 224)
pil_resized = pil_image.resize((IMG_SIZE, IMG_SIZE))
```

⚠️ **`unsqueeze(0)`** : `timm` attend un batch (dim `B` en tête). On passe de `(3, 224, 224)` à `(1, 3, 224, 224)`.

</details>


In [ ]:
# TODO

## Exercice 2 — Capturer et visualiser l'attention par tête

Le but est de récupérer la matrice d'attention `softmax(QK^T/√d_k)` du **dernier bloc**, puis d'afficher
les `H=6` cartes par tête en overlay sur l'image.

1. Écrivez une fonction `install_attention_capture(attn_module, store)` qui monkey-patche le forward de `attn_module` pour stocker `attn` (de shape `(B, H, N, N)`) dans le dict `store`.
2. Installez-la sur `model.blocks[-1].attn`, faites le forward sous `torch.no_grad()`, restaurez le forward original.
3. Récupérez `attn[0, :, 0, 1:]` (attention du `[CLS]` vers les patches), reshape en `(H, 14, 14)`.
4. Upsamplez à `(H, 224, 224)` via `torch.nn.functional.interpolate(..., mode="bilinear")`.
5. Affichez `H=6` cartes par tête + leur moyenne, en overlay sur l'image. Observez la spécialisation.

<details>
<summary>💡 Coup de pouce — forward patché et extraction CLS-vers-patches</summary>

**🎯 But :** intercepter `softmax(QK^T/√d_k)` proprement, sans réécrire tout le modèle.

**Le forward patché : copie de celui de timm avec un `store`**

```python
def install_attention_capture(attn_module, store: dict):
    orig_forward = attn_module.forward

    def patched(x):
        B, N, C = x.shape
        qkv = attn_module.qkv(x).reshape(B, N, 3, attn_module.num_heads, attn_module.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)                  # (3, B, H, N, head_dim)
        q, k, v = qkv.unbind(0)
        if hasattr(attn_module, "q_norm"):
            q, k = attn_module.q_norm(q), attn_module.k_norm(k)
        attn = (q @ k.transpose(-2, -1)) * attn_module.scale   # (B, H, N, N)
        attn = attn.softmax(dim=-1)
        store["attn"] = attn.detach().cpu()                # ← capture ici
        attn = attn_module.attn_drop(attn)
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return attn_module.proj_drop(attn_module.proj(out))

    attn_module.forward = patched
    return lambda: setattr(attn_module, "forward", orig_forward)
```

**Forward, restauration, extraction**

```python
store = {}
restore = install_attention_capture(model.blocks[-1].attn, store)
with torch.no_grad():
    _ = model(x)
restore()                                                   # ← TRÈS important

attn = store["attn"][0]                                     # (H, N, N), N=197
H, N, _ = attn.shape
n_patches = N - 1                                           # 196 patches
grid = int(round(n_patches ** 0.5))                         # 14
cls_attn = attn[:, 0, 1:].reshape(H, grid, grid)            # (H, 14, 14)
```

⚠️ **Pourquoi `[:, 0, 1:]`** ? Ligne `0` = c'est le token `[CLS]` qui regarde les autres. Colonnes `1:` = les 196 patches (on saute le `[CLS]` lui-même). Résultat : poids d'attention du `[CLS]` vers chaque patch.

**Upsampler à 224x224**

```python
import torch.nn.functional as F
cls_attn_up = F.interpolate(
    cls_attn.unsqueeze(1),                                  # (H, 1, 14, 14)
    size=(224, 224),
    mode="bilinear",
    align_corners=False,
)[:, 0].numpy()                                             # (H, 224, 224)
```

**Visualiser : image + 6 têtes + moyenne**

```python
fig, axes = plt.subplots(2, 4, figsize=(11, 5.5))
axes[0, 0].imshow(pil_resized); axes[0, 0].set_title("Image"); axes[0, 0].axis("off")
for i in range(6):
    row, col = divmod(i, 3)
    ax = axes[row, col + 1] if row == 0 else axes[1, i - 3]
    ax.imshow(pil_resized, alpha=0.55)
    ax.imshow(cls_attn_up[i], cmap="inferno", alpha=0.55)
    ax.set_title(f"head {i}"); ax.axis("off")
axes[1, 3].imshow(pil_resized, alpha=0.55)
axes[1, 3].imshow(cls_attn_up.mean(axis=0), cmap="inferno", alpha=0.55)
axes[1, 3].set_title("Moyenne", fontweight="bold"); axes[1, 3].axis("off")
plt.tight_layout(); plt.show()
```

**Ce que vous devriez observer**

Chaque tête a son propre pattern : certaines suivent le contour de l'objet, d'autres se concentrent sur des régions discriminantes (œil, museau pour un animal), d'autres regardent le fond. La moyenne donne une carte plus globale qui isole approximativement l'objet principal.

⚠️ **Différence avec DINO (mention culturelle)** : un ViT pré-entraîné en **self-supervision** (DINO) a souvent des cartes qui isolent l'objet bien plus nettement, sans label. C'est l'un des résultats marquants de 2021. Ici on utilise un ViT **supervisé ImageNet** : les têtes se concentrent sur des indices discriminants, mais ne segmentent pas vraiment.

</details>


In [ ]:
# TODO

## Exercice 3 — Carte de régions d'intérêt par seuillage

Idée : la moyenne des attentions du `[CLS]` désigne les régions sur lesquelles le ViT appuie sa décision.
On peut la seuiller pour produire une carte binaire des « régions importantes ».

1. Calculez la moyenne sur les têtes : `cls_attn_up.mean(axis=0)`.
2. Normalisez l'image entre 0 et 1 (`(x - x.min()) / (x.max() - x.min())`).
3. Gardez les `keep=60%` des pixels les plus actifs via un quantile.
4. Affichez côte à côte : image, attention moyenne, masque seuil.

<details>
<summary>💡 Coup de pouce — seuillage par quantile</summary>

**🎯 But :** transformer la carte d'attention continue en masque binaire interprétable.

**Normaliser + seuiller**

```python
def attention_to_mask(cls_attn_up, keep=0.6):
    avg = cls_attn_up.mean(axis=0)
    avg = (avg - avg.min()) / (avg.max() - avg.min() + 1e-8)
    threshold = np.quantile(avg, 1 - keep)
    mask = (avg >= threshold).astype(np.float32)
    return mask, avg
```

⚠️ **`np.quantile(avg, 1 - keep)`** : si `keep=0.6`, on prend le quantile à 0.4 (60% des pixels ont une valeur > ce quantile). C'est exactement « garder les 60% les plus actifs ».

**Affichage**

```python
mask, avg = attention_to_mask(cls_attn_up, keep=0.5)
fig, axes = plt.subplots(1, 3, figsize=(11, 4))
axes[0].imshow(pil_resized); axes[0].set_title("Image"); axes[0].axis("off")
axes[1].imshow(pil_resized, alpha=0.55); axes[1].imshow(avg, cmap="inferno", alpha=0.55)
axes[1].set_title("Attention moyenne"); axes[1].axis("off")
axes[2].imshow(pil_resized, alpha=0.5); axes[2].imshow(mask, cmap="Reds", alpha=0.45)
axes[2].set_title("Régions d'intérêt"); axes[2].axis("off")
plt.tight_layout(); plt.show()
```

⚠️ **Ce n'est PAS une vraie segmentation.** Un ViT supervisé ImageNet se concentre sur des indices discriminants (parfois locaux). Pour une vraie segmentation émergente, il faudrait un ViT **self-supervised** (DINO).

</details>


In [ ]:
# TODO

## Exercice 4 — Bonus : correspondance dense entre patches

Idée : choisir un patch dans l'image, et trouver les patches **les plus similaires** dans la même image.
On utilise les représentations des patches en sortie du modèle (vecteurs `D=384` par patch).

1. Récupérez les features par patch via `model.forward_features(x)` (timm). Shape `(1, N=197, D=384)`.
2. Skippez le token `[CLS]` : `feats[0, 1:]` de shape `(196, 384)`.
3. Reshape en `(14, 14, 384)` pour avoir une grille spatiale.
4. Choisissez un patch query par ses coordonnées `(qi, qj)` (ex : `(7, 7)` = centre).
5. Calculez la similarité cosinus entre ce patch et tous les autres.
6. Reshape en `(14, 14)`, upsample, affichez en overlay avec un cadre rouge sur le patch query.

<details>
<summary>💡 Coup de pouce — similarité cosinus entre patches</summary>

**🎯 But :** montrer que les features par patch capturent une notion de **ressemblance sémantique** : des patches d'un même objet sont proches dans l'espace ViT.

**Extraire les features par patch**

```python
with torch.no_grad():
    feats = model.forward_features(x)                     # (1, 197, 384)
patches = feats[0, 1:]                                     # (196, 384) — skip CLS
grid = int(round(patches.shape[0] ** 0.5))                 # 14
patches_2d = patches.reshape(grid, grid, -1)               # (14, 14, 384)
```

⚠️ **`forward_features`** : c'est le forward de timm qui s'arrête juste avant la tête de classification. Idéal pour avoir les features intermédiaires.

**Similarité cosinus avec un patch query**

```python
qi, qj = 7, 7                                              # centre
q = patches_2d[qi, qj]                                     # (384,)
sim = torch.nn.functional.cosine_similarity(
    patches, q.unsqueeze(0).expand_as(patches), dim=-1
).reshape(grid, grid).numpy()                              # (14, 14)
```

⚠️ **Similarité cosinus vs distance L2** : sur des features de grande dimension, le cosinus est en général plus stable et interprétable (entre -1 et 1).

**Upsample + affichage**

```python
import torch.nn.functional as F
sim_up = F.interpolate(
    torch.from_numpy(sim).unsqueeze(0).unsqueeze(0),
    size=(224, 224), mode="bilinear", align_corners=False,
)[0, 0].numpy()

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(pil_resized)
patch_size = 224 // grid                                   # 16
axes[0].add_patch(plt.Rectangle((qj * patch_size, qi * patch_size),
                                 patch_size, patch_size,
                                 fill=False, edgecolor="red", linewidth=2))
axes[0].set_title(f"Patch query ({qi},{qj})"); axes[0].axis("off")
axes[1].imshow(pil_resized, alpha=0.5); axes[1].imshow(sim_up, cmap="viridis", alpha=0.55)
axes[1].set_title("Similarité cosinus"); axes[1].axis("off")
plt.tight_layout(); plt.show()
```

**Ce que vous devriez observer**

Si vous prenez un patch sur l'objet (par ex sur la tête d'un chat), les patches les plus similaires sont sur le **reste de l'objet** (le corps du chat), pas sur le fond. C'est le signe que les features ont appris une représentation **structurée par objet**.

</details>


In [ ]:
# TODO